# N05: Museum B, scored once

The test corpus: 600 paintings from 44 artists never seen in dev. Nothing is tuned here. Every rung runs with the
frozen config, their top 5 are pooled and judged **in one blind pass** (all rungs mixed, shuffled), then scored.

Absolute scores are not comparable with dev (different paintings, 600 vs 480): compare the **gaps between rungs**.

In [ ]:
import pandas as pd

from deep_learning_project_waerniers_lamy.nlp import utils as u

pd.set_option("display.width", 160)
from pathlib import Path

In [ ]:
frozen = u.check_frozen()
paintings = u.load_corpus("test")
descriptions = u.load_descriptions()
queries = u.QUERIES["text"].tolist()
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
    "R2 description": u.rung_description(paintings, queries, descriptions),
    "R3 description+metadata": u.rung_description_metadata(paintings, queries, descriptions),
    "R4 siglip image": u.rung_siglip(paintings, queries),
    "R5 hybrid": u.rung_hybrid(paintings, queries, descriptions),
}
print("artist named in test descriptions:", round(u.artist_leak_rate(paintings, descriptions), 3))
u.write_judging_page("test", scores, paintings)

### Judging (by hand)

Open the page printed above in a browser. It shows **only the query and the image** (no rung name, artist, style or
description), and only paintings not judged before. Grade each one 0 / 1 / 2 with the keyboard, click *export JSON*,
then run the next cell. Grades are kept in the browser, so you can stop and come back.

In [ ]:
u.import_judgments("test", Path.home() / "Downloads" / "judgments_test.json")

## Results

In [ ]:
per_query = u.per_query_scores("test", scores, paintings)
table = u.summary_table(per_query)
u.plot_ladder(table, "Museum B (test): full ladder")
display(table)
u.compare(per_query, [("R2 description", "R1 metadata"), ("R4 siglip image", "R1 metadata"),
                      ("R5 hybrid", frozen["best_single"])])

## Gaps to the baseline, dev vs Museum B

In [ ]:
dev_table = pd.read_csv(u.RESULTS_DIR / "dev_summary.csv").set_index("rung")["nDCG@5"]
test_table = table.set_index("rung")["nDCG@5"]
pd.DataFrame({"dev gap to R1": dev_table - dev_table["R1 metadata"],
              "test gap to R1": test_table - test_table["R1 metadata"]}).round(3)

In [ ]:
table.to_csv(u.RESULTS_DIR / "test_summary.csv", index=False)
per_query.to_csv(u.RESULTS_DIR / "test_per_query.csv", index=False)